# Basira — train a Kraken Arabic **handwriting** model (Muharaf)

**Before running:** Runtime → Change runtime type → **GPU (T4)**.

Transfer-learns from the OpenITI Arabic base model on `aamijar/muharaf-public`
(24,495 line image↔text pairs). Output: a `.safetensors` model in `muharaf_hw/`.
~1–3 h on a free T4. (Verified against **Kraken 7.x**.)

⚠️ **Licence:** Muharaf is **CC BY-NC-SA (non-commercial)** — R&D / demo use only.


In [ ]:
# 1) Install Kraken + check the GPU
!pip install -q kraken datasets pillow
import torch
print("CUDA available:", torch.cuda.is_available())   # must print True
# If pip says "restart": Runtime > Restart session, then re-run from here.


In [ ]:
# 2) Build Kraken training pairs (line image + .gt.txt) from the dataset
import os
from datasets import load_dataset

ds = load_dataset("aamijar/muharaf-public")           # public; no token needed
# QUICK TEST first? Uncomment to train on a small subset (~20 min):
# ds["train"] = ds["train"].select(range(2000))

for split in ds:                                       # train / validation / test
    out = f"data/{split}"; os.makedirs(out, exist_ok=True); n = 0
    for i, row in enumerate(ds[split]):
        text = (row.get("text") or "").strip()
        if not text:
            continue
        img = row["image"]
        if img.mode not in ("RGB", "L"):
            img = img.convert("RGB")
        img.save(f"{out}/{i:06d}.png")
        with open(f"{out}/{i:06d}.gt.txt", "w", encoding="utf-8") as fh:
            fh.write(text)
        n += 1
    print(split, "->", n, "line pairs")


In [ ]:
# 3) Fetch the OpenITI Arabic base model (for transfer learning)
!kraken get 10.5281/zenodo.7050270
import glob, os
base = glob.glob(os.path.expanduser("~/.local/share/htrmopo/**/*.mlmodel"), recursive=True)[0]
print("base model:", base)


In [ ]:
# 4) Train. Kraken 7: (a) compile to a binary dataset; (b) --force-type baseline
# matches the line-strip data to the baseline base model; (c) -o is a DIRECTORY.
import glob, os
base = glob.glob(os.path.expanduser("~/.local/share/htrmopo/**/*.mlmodel"), recursive=True)[0]
!ketos compile -f path --force-type baseline -o muharaf_train.arrow data/train/*.png
cmd = f'ketos -d cuda:0 train -f binary --resize union -i "{base}" -o muharaf_hw muharaf_train.arrow'
print(cmd)
!{cmd}
# Watch val_accuracy each epoch — that's held-out character accuracy
# (CER ~= 1 - val_accuracy). Training auto-stops when it plateaus.


In [ ]:
# 5) Locate the best trained model (Kraken 7 writes .safetensors into muharaf_hw/)
import glob
models = sorted(glob.glob("muharaf_hw/best_*.safetensors")) or sorted(glob.glob("muharaf_hw/*.safetensors"))
print("best model:", models[-1] if models else "NONE - check training output above")


In [ ]:
# 6) Save the model to Google Drive (then download from drive.google.com)
import glob, shutil
models = sorted(glob.glob("muharaf_hw/best_*.safetensors")) or sorted(glob.glob("muharaf_hw/*.safetensors"))
best = models[-1]
from google.colab import drive
drive.mount("/content/drive")
shutil.copy(best, "/content/drive/MyDrive/muharaf_hw_best.safetensors")
print("Saved to Drive as muharaf_hw_best.safetensors  (from", best + ")")


## Back on your Mac
Download `muharaf_hw_best.safetensors` from Google Drive, then with the app +
Kraken sidecar running:
```bash
scripts/install-kraken-model.sh ~/Downloads/muharaf_hw_best.safetensors muharaf
```
Set `DEMO_TRANSCRIBE_ADAPTER=kraken-muharaf` in `.env`, restart (`pnpm dev`),
log in as `demo@basira.test`, and transcribe — it runs your handwriting model.
